# Final ET products — Fundación

Read-only local visualizer for the minimal final outputs. This notebook does not train models, query Earth Engine, reconcile ET, or write derived figures/tables.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

FINAL_DATES = ['2020-03-13', '2021-11-25', '2022-03-30']

def locate_final_dir():
    start = Path.cwd().resolve()
    candidates = [start, start / 'final']
    for parent in [start, *start.parents]:
        candidates.extend([
            parent / 'ET_fundacion_workspace' / 'final',
            parent / 'final',
        ])
    for candidate in candidates:
        if (candidate / 'raster_summary.csv').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate ET_fundacion_workspace/final.')

FINAL_DIR = locate_final_dir()
FINAL_DIR


## Raster summary


In [ ]:
summary = pd.read_csv(FINAL_DIR / 'raster_summary.csv')
summary


## ET maps

Each map is the exact band-1 ET field from the corresponding frozen scientific multiband raster. Display scaling below uses the 1st–99th percentiles only for visualization; raster values are not modified.


In [ ]:
for date in FINAL_DATES:
    path = FINAL_DIR / f'ET_{date}_20m.tif'
    with rasterio.open(path) as src:
        et = src.read(1, masked=True)
        values = et.compressed()
        vmin, vmax = np.percentile(values, [1, 99])
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(et, extent=extent, vmin=vmin, vmax=vmax)
    ax.set_title(f'ET — {date}')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label('ET (mm per MODIS period)')
    plt.show()


## Published vs common spatial support


In [ ]:
summary.loc[:, [
    'date', 'scope', 'valid_pixels', 'area_km2',
    'fraction_of_published_support', 'ET_median_mm_period',
    'ET_mean_mm_period', 'ET_p05_mm_period', 'ET_p95_mm_period'
]]
